# Isolate the 114 unanimously included batch-A cases

Select a case **only when Becca, Hyunsang, and Yinkui each mark
`eligibility.binary_task == "include"`**. Join by `case_id`, never row position.

Run all cells in order with Python 3; only the standard library is required.
Launch from the repository root or this `MVP/` directory. The notebook
reads the three original submissions and the source records without changing them.

This is an eligibility selection, not a human-reviewed modeling dataset.
Agreement on eligibility does not establish agreement on target labels.
Source text is preserved verbatim; it is not yet FULL or MASKED text.
Family review, calibration flags, splits, cleaning, masking, and target review
are outside this notebook's scope.


In [1]:
from pathlib import Path
from collections import Counter
import hashlib
import json
import subprocess

REVIEWERS = ("becca", "hyunsang", "yinkui")
EXPECTED_BATCH_COUNT = 169
EXPECTED_SELECTED_COUNT = 114
VERSION = "unanimous_include_v1"

ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "annotation/source/cases.jsonl").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run from the legal-judgment repository or the MVP directory.")
MVP_DIR = ROOT / "annotation/submissions/becca/MVP"
OUTPUT_DIR = MVP_DIR / "data" / VERSION
INPUT_PATHS = {
    **{name: ROOT / f"annotation/submissions/{name}/batches/A/annotations.jsonl"
       for name in REVIEWERS},
    "batch_manifest": ROOT / "annotation/batches/A/manifest.json",
    "sources": ROOT / "annotation/source/cases.jsonl",
}

def require(condition, message):
    if not condition:
        raise ValueError(message)

def read_records(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()
            if line.strip()]

def index_unique(rows, name):
    ids = [row["case_id"] for row in rows]
    require(all(isinstance(cid, str) and cid for cid in ids), f"Invalid IDs in {name}")
    require(len(ids) == len(set(ids)), f"Duplicate case IDs in {name}")
    return dict(zip(ids, rows))

print("Selection version:", VERSION)


Selection version: unanimous_include_v1


## 1. Validate the reviewers' case coverage

Every reviewer must cover the same 169 unique IDs listed in the batch-A manifest.
Missing cases or duplicate IDs stop execution instead of being silently dropped.


In [2]:
batch_manifest = json.loads(INPUT_PATHS["batch_manifest"].read_text(encoding="utf-8"))
batch_ids = batch_manifest["case_ids"]
require(len(batch_ids) == len(set(batch_ids)) == EXPECTED_BATCH_COUNT,
        "Unexpected or duplicate batch-A IDs")
reviews = {}
for name in REVIEWERS:
    reviews[name] = index_unique(read_records(INPUT_PATHS[name]), name)
    require(set(reviews[name]) == set(batch_ids), f"Case coverage mismatch: {name}")
    decisions = [row["eligibility"]["binary_task"] for row in reviews[name].values()]
    require(set(decisions) <= {"include", "exclude", "review"},
            f"Unrecognized eligibility decision: {name}")
    counts = Counter(decisions)
    print(f"{name:8s}: " + ", ".join(f"{value}={counts[value]}"
                                      for value in ("include", "exclude", "review")))


becca   : include=119, exclude=11, review=39
hyunsang: include=132, exclude=13, review=24
yinkui  : include=122, exclude=11, review=36


## 2. Select unanimous inclusions and retain the complete audit

The expected 114-case count is a check, not a sampling quota. If updated inputs
change the selection, investigate and create a new dataset version.


In [3]:
eligibility_audit = []
for case_id in sorted(batch_ids):
    decisions = {name: reviews[name][case_id]["eligibility"]["binary_task"]
                 for name in REVIEWERS}
    eligibility_audit.append({
        "case_id": case_id,
        **{f"{name}_eligibility": decision for name, decision in decisions.items()},
        "unanimous_include": all(value == "include" for value in decisions.values()),
        "eligibility_agreement": len(set(decisions.values())) == 1,
    })

selected_ids = [row["case_id"] for row in eligibility_audit if row["unanimous_include"]]
require(len(selected_ids) == EXPECTED_SELECTED_COUNT,
        f"Expected {EXPECTED_SELECTED_COUNT} unanimous inclusions; found {len(selected_ids)}")
patterns = Counter(tuple(row[f"{name}_eligibility"] for name in REVIEWERS)
                   for row in eligibility_audit)
print("Unanimous include:", len(selected_ids))
print("Unanimous exclude:", patterns[("exclude",) * 3])
print("Unanimous review:", patterns[("review",) * 3])
print("Eligibility disagreements:", sum(not row["eligibility_agreement"]
                                        for row in eligibility_audit))
print("Total audited:", len(eligibility_audit))


Unanimous include: 114
Unanimous exclude: 9
Unanimous review: 19
Eligibility disagreements: 27
Total audited: 169


## 3. Link selected IDs to the original source opinions

Preserve each selected source record in full, including its text, URLs, metadata,
and hash. Verify that all three annotations refer to that same text hash.
Annotation fields (outcomes, findings, summaries, and quotations) are not merged
into the source export. Empty source text stops execution.


In [4]:
sources = index_unique(read_records(INPUT_PATHS["sources"]), "sources")
require(set(selected_ids) <= set(sources), "Selected cases missing from source records")
selected_cases = [sources[case_id] for case_id in selected_ids]
for case in selected_cases:
    cid = case["case_id"]
    require(bool(case.get("source_url")), f"Missing source URL: {cid}")
    require(isinstance(case.get("text"), str) and bool(case["text"].strip()),
            f"Empty source text: {cid}")
    actual_hash = hashlib.sha256(case["text"].encode("utf-8")).hexdigest()
    require(actual_hash == case.get("text_sha256"), f"Source hash mismatch: {cid}")
    for name in REVIEWERS:
        require(reviews[name][cid]["text_sha256"] == actual_hash,
                f"Annotation/source hash mismatch: {name}, {cid}")
print(f"Verified {len(selected_cases)} unique source records, nonempty texts, and reviewer hashes.")
print("First 10 selected case IDs:")
print("\n".join(selected_ids[:10]))


Verified 114 unique source records, nonempty texts, and reviewer hashes.
First 10 selected case IDs:
NY-10090934
NY-10128963
NY-10160793
NY-10291100
NY-10309802
NY-2682947
NY-2686388
NY-2686408
NY-2719389
NY-2757329


## 4. Export the isolated cases and reproducibility record

Outputs go to `data/unanimous_include_v1/` inside this `MVP/` folder:

- `case_ids.json`: the fixed, sorted list of 114 selected IDs.
- `cases.jsonl`: the 114 original source records, in the same ID order.
- `eligibility_audit.jsonl`: all 169 cases with the three decisions and selection flag.
- `selection_manifest.json`: rule, counts, input/output checksums, and repository revision.

Re-running with unchanged inputs is safe. Existing output files with different
contents cause an error; use a new version rather than overwrite a prior selection.
The manifest records an eligibility subset, not a train/development/test split.


In [5]:
def json_bytes(value):
    return (json.dumps(value, ensure_ascii=False, indent=2) + "\n").encode("utf-8")

def jsonl_bytes(rows):
    return "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in rows).encode("utf-8")

payloads = {
    "case_ids.json": json_bytes(selected_ids),
    "cases.jsonl": jsonl_bytes(selected_cases),
    "eligibility_audit.jsonl": jsonl_bytes(eligibility_audit),
}
try:
    repository_commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=ROOT, text=True, stderr=subprocess.DEVNULL
    ).strip()
except (OSError, subprocess.CalledProcessError):
    repository_commit = None

selection_manifest = {
    "version": VERSION,
    "batch": "A",
    "selection_rule": "All three reviewers have eligibility.binary_task == include",
    "reviewers": list(REVIEWERS),
    "audit_count": len(eligibility_audit),
    "selected_count": len(selected_ids),
    "case_ids": selected_ids,
    "review_status": "Unanimous inclusion only; human review and target agreement not established",
    "text_status": "Original source text; no cleaning or masking applied",
    "repository_commit_at_run": repository_commit,
    "inputs": {
        name: {"path": str(path.relative_to(ROOT)),
               "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
        for name, path in INPUT_PATHS.items()
    },
    "outputs": {name: {"sha256": hashlib.sha256(content).hexdigest()}
                for name, content in payloads.items()},
}
# Preserve the creation revision when inputs and output content have not changed.
manifest_path = OUTPUT_DIR / "selection_manifest.json"
if manifest_path.exists():
    previous_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    comparable = {k: v for k, v in selection_manifest.items() if k != "repository_commit_at_run"}
    previous_comparable = {k: v for k, v in previous_manifest.items()
                           if k != "repository_commit_at_run"}
    require(comparable == previous_comparable,
            "Selection manifest changed. Investigate and choose a new VERSION.")
    selection_manifest = previous_manifest
payloads["selection_manifest.json"] = json_bytes(selection_manifest)

# Check every destination before writing any output.
for filename, content in payloads.items():
    destination = OUTPUT_DIR / filename
    require(not destination.exists() or destination.read_bytes() == content,
            f"Existing output differs: {filename}. Choose a new VERSION.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for filename, content in payloads.items():
    destination = OUTPUT_DIR / filename
    if not destination.exists():
        destination.write_bytes(content)

exported_ids = json.loads((OUTPUT_DIR / "case_ids.json").read_text(encoding="utf-8"))
exported_cases = read_records(OUTPUT_DIR / "cases.jsonl")
require(exported_ids == selected_ids == [row["case_id"] for row in exported_cases],
        "Exported IDs and source records do not match")
require(exported_cases == selected_cases, "Exported source records changed")
for filename in payloads:
    print("Saved:", (OUTPUT_DIR / filename).relative_to(ROOT))
print(f"Complete: {len(exported_ids)} cases isolated; {len(eligibility_audit)} audit rows retained.")


Saved: annotation/submissions/becca/MVP/data/unanimous_include_v1/case_ids.json
Saved: annotation/submissions/becca/MVP/data/unanimous_include_v1/cases.jsonl
Saved: annotation/submissions/becca/MVP/data/unanimous_include_v1/eligibility_audit.jsonl
Saved: annotation/submissions/becca/MVP/data/unanimous_include_v1/selection_manifest.json
Complete: 114 cases isolated; 169 audit rows retained.
